<a href="https://www.kaggle.com/code/fiftythirtyfour/predicting-irrigation-need-0-77581?scriptVersionId=311372519" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Load

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')

In [ ]:
df.head()

In [ ]:
num = df.select_dtypes('number').columns.to_list()
cat = df.select_dtypes('object').columns.to_list()
target = 'Irrigation_Need'
cat.remove(target)
num.remove('id')

In [ ]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=.2)

# Data Science

In [ ]:
df[target].value_counts()

In [ ]:
cat

In [ ]:
import seaborn as sns

sns.heatmap(
    df.assign(is_low = (df['Irrigation_Need'] == 'Low').astype(int))
      .groupby(['Soil_Type', 'Region'])['is_low']
      .mean()
      .unstack(),
    annot=True,
    fmt=".3f"
)

# Machine

In [ ]:
cat

In [ ]:
num

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

num = ['Soil_Moisture', 'Temperature_C', 'Wind_Speed_kmh', 'Rainfall_mm']
cat = ['Crop_Growth_Stage', 'Mulching_Used']

processor = make_column_transformer(
    (make_pipeline(SimpleImputer(), StandardScaler()), num)
    , (OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat)
)

train_pre = processor.fit_transform(train)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

log = LogisticRegression(max_iter=1000)
log.fit(train_pre, train[target])
cross_val_score(log, train_pre, train[target]).mean()

In [ ]:
model = log
pipe = make_pipeline(processor, model)

In [ ]:
accuracy_score(test[target], pipe.predict(test))

# Submission

In [ ]:
samp = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')
pd.DataFrame({
    'id': samp['id']
    , 'Irrigation_Need': pipe.predict(samp)
}).to_csv('submission.csv', index=False)